# Matched Three-Method and Exclusion Sensitivity


In [1]:
NOTEBOOK_STEM = "01_matched_three_method_exclusion_sensitivity"


Objective: verify that final cross-method statistics use the same matched sample set where
FTIR EC, HIPS Fabs, and aethalometer IR BC are all present, then quantify how manual and
threshold exclusions change slope/intercept/R2.


In [2]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REPO_ROOT = Path("/Users/ahmadjalil/github/aethmodular")
FTIR_DIR = REPO_ROOT / "research" / "ftir_hips_chem"
CATCH_UP_DIR = REPO_ROOT / "research" / "catch_up"
DATA_ROOT = Path('/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data')
OUT_DIR = CATCH_UP_DIR / "output" / globals().get("NOTEBOOK_STEM", Path.cwd().name)
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCRIPTS_DIR = FTIR_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from config import SITES
from outliers import apply_exclusion_flags, apply_threshold_flags, get_clean_data
from plotting import PlotConfig, apply_default_style

apply_default_style()
PlotConfig.set(sites="all", layout="individual", show_stats=True, show_1to1=True)

SITE_CODES = {site: cfg["code"] for site, cfg in SITES.items()}
CODE_TO_SITE = {v: k for k, v in SITE_CODES.items()}
SITE_COLORS = {site: PlotConfig.get_site_color(site) for site in SITE_CODES}

PARAM_RENAME = {
    "EC_ftir": "ftir_ec",
    "OC_ftir": "ftir_oc",
    "HIPS_Fabs": "hips_fabs",
    "HIPS_T1": "hips_t1",
    "HIPS_R1": "hips_r1",
    "HIPS_t": "hips_t",
    "HIPS_r": "hips_r",
    "HIPS_tau": "hips_tau",
    "ChemSpec_EC_PM2.5": "chemspec_ec",
    "ChemSpec_OC_PM2.5": "chemspec_oc",
    "ChemSpec_OM_PM2.5": "chemspec_om",
    "ChemSpec_Iron_PM2.5": "iron",
    "ChemSpec_Silicon_PM2.5": "silicon",
    "ChemSpec_Aluminum_PM2.5": "aluminum",
    "ChemSpec_Calcium_PM2.5": "calcium",
    "ChemSpec_Titanium_PM2.5": "titanium",
    "ChemSpec_Filter_PM2.5_mass": "pm25_mass",
}

def _first_existing(paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    raise FileNotFoundError("None of these paths exist:\n" + "\n".join(map(str, paths)))

def load_filter_long():
    path = _first_existing([
        FTIR_DIR / "Filter Data" / "unified_filter_dataset.pkl",
        DATA_ROOT / "Combine csv files" / "Filter Data" / "unified_filter_dataset.pkl",
        DATA_ROOT / "Combine csv files" / "FTIR_HIPS_Chem" / "Filter Data" / "unified_filter_dataset.pkl",
    ])
    df = pd.read_pickle(path)
    df["SampleDate"] = pd.to_datetime(df["SampleDate"])
    df["base_filter_id"] = df["FilterId"].astype(str).str.replace(r"-\d+$", "", regex=True)
    print(f"Loaded filter data: {path}  rows={len(df):,}")
    return df

def load_filter_wide(params):
    long = load_filter_long()
    d = long[long["Parameter"].isin(params)].copy()
    meta = (
        d.sort_values(["Site", "base_filter_id", "SampleDate"])
         .groupby(["Site", "base_filter_id"], as_index=False)
         .agg(
             filter_id=("FilterId", "first"),
             date=("SampleDate", "first"),
             volume_m3=("Volume_m3", "max"),
             deposit_area_cm2=("DepositArea_cm2", "max"),
             lot_id=("LotId", "first"),
         )
    )
    conc = d.pivot_table(
        index=["Site", "base_filter_id"],
        columns="Parameter",
        values="Concentration",
        aggfunc="first",
    ).rename(columns=PARAM_RENAME)
    mass = d.pivot_table(
        index=["Site", "base_filter_id"],
        columns="Parameter",
        values="MassLoading_ug",
        aggfunc="first",
    ).rename(columns={p: PARAM_RENAME.get(p, p) + "_mass_ug" for p in params})
    wide = meta.merge(conc.reset_index(), on=["Site", "base_filter_id"], how="left")
    wide = wide.merge(mass.reset_index(), on=["Site", "base_filter_id"], how="left")
    wide["site"] = wide["Site"].map(CODE_TO_SITE)
    return wide

def load_aeth_site(site):
    file_map = {
        "Beijing": "df_Beijing_9am_resampled.pkl",
        "Delhi": "df_Delhi_9am_resampled.pkl",
        "JPL": "df_JPL_9am_resampled.pkl",
        "Addis_Ababa": "df_Addis_Ababa_9am_resampled.pkl",
    }
    repo_path = FTIR_DIR / "processed_sites" / file_map[site]
    cloud_candidates = [
        DATA_ROOT / "Aethelometry Data" / "JacrosMA350 60s Data20250804082112" / "df_Jacros_9am_resampled.pkl",
        DATA_ROOT / "Aethelometry Data" / "Kyan Data" / "Dataset" / "df_cleaned_Beijing_manual_BCc.pkl",
        DATA_ROOT / "Aethelometry Data" / "Kyan Data" / "Dataset" / "df_cleaned_Delhi_manual_BCc.pkl",
        DATA_ROOT / "Aethelometry Data" / "Kyan Data" / "Dataset" / "df_cleaned_JPL_manual_BCc.pkl",
    ]
    path = repo_path if repo_path.exists() else _first_existing([p for p in cloud_candidates if site.replace("_Ababa", "") in p.name or site == "Addis_Ababa"])
    df = pd.read_pickle(path)
    df = df.loc[:, ~df.columns.duplicated()].copy()
    if "day_9am" in df.columns:
        df["date"] = pd.to_datetime(df["day_9am"]).dt.normalize()
    elif "datetime_local" in df.columns:
        df["date"] = pd.to_datetime(df["datetime_local"]).dt.normalize()
    else:
        df["date"] = pd.to_datetime(df.index).normalize()
    df["site"] = site
    return df

def _to_ugm3(s):
    s = pd.to_numeric(s, errors="coerce")
    med = s.dropna().abs().median()
    return s / 1000.0 if pd.notna(med) and med > 100 else s

def aeth_metrics(site):
    df = load_aeth_site(site)
    out = df[["site", "date"]].copy()
    for wl in ["UV", "Blue", "Green", "Red", "IR"]:
        col = f"{wl} BCc"
        out[f"{wl.lower()}_bc_ugm3"] = _to_ugm3(df[col]) if col in df.columns else np.nan
    out["aeth_ir_ugm3"] = out["ir_bc_ugm3"]
    out["uv_ir_bcc_ratio"] = out["uv_bc_ugm3"] / out["ir_bc_ugm3"]
    out["green_ir_bcc_ratio"] = out["green_bc_ugm3"] / out["ir_bc_ugm3"]
    out["delta_c_ugm3"] = out["uv_bc_ugm3"] - out["ir_bc_ugm3"]
    # Raw attenuation-based apparent AAE from rolling mean dATN when available.
    uv = pd.to_numeric(df.get("delta UV ATN1 rolling mean", np.nan), errors="coerce")
    ir = pd.to_numeric(df.get("delta IR ATN1 rolling mean", np.nan), errors="coerce")
    mask = (uv > 0) & (ir > 0)
    out["aae_atn_uv_ir"] = np.where(mask, -np.log(uv / ir) / np.log(375 / 880), np.nan)
    return out.groupby(["site", "date"], as_index=False).median(numeric_only=True)

def matched_dataset(include_chem=True):
    params = ["EC_ftir", "OC_ftir", "HIPS_Fabs", "HIPS_T1", "HIPS_R1", "HIPS_t", "HIPS_r", "HIPS_tau"]
    if include_chem:
        params += [
            "ChemSpec_EC_PM2.5", "ChemSpec_OC_PM2.5", "ChemSpec_OM_PM2.5",
            "ChemSpec_Iron_PM2.5", "ChemSpec_Silicon_PM2.5", "ChemSpec_Aluminum_PM2.5",
            "ChemSpec_Calcium_PM2.5", "ChemSpec_Titanium_PM2.5", "ChemSpec_Filter_PM2.5_mass",
        ]
    filt = load_filter_wide(params)
    filt["date"] = pd.to_datetime(filt["date"]).dt.normalize()
    aeth = pd.concat([aeth_metrics(s) for s in SITE_CODES], ignore_index=True)
    m = filt.merge(aeth, on=["site", "date"], how="left")
    # Units: ChemSpec metals are usually ng/m3; convert common tracers to ug/m3 for ratios.
    for col in ["iron", "silicon", "aluminum", "calcium", "titanium"]:
        if col in m.columns:
            med = pd.to_numeric(m[col], errors="coerce").dropna().abs().median()
            if pd.notna(med) and med > 50:
                m[col + "_ugm3"] = m[col] / 1000.0
            else:
                m[col + "_ugm3"] = m[col]
    m["ec_mass_ug"] = m.get("ftir_ec_mass_ug")
    if "ftir_ec" in m.columns:
        m["ec_mass_from_volume_ug"] = m["ftir_ec"] * m["volume_m3"]
        m["ec_mass_ug"] = m["ec_mass_ug"].fillna(m["ec_mass_from_volume_ug"])
    m["ec_surface_loading_ug_cm2"] = m["ec_mass_ug"] / m["deposit_area_cm2"]
    m["hips_bc_mac10_ugm3"] = m["hips_fabs"] / 10.0
    m["hips_minus_ftir"] = m["hips_bc_mac10_ugm3"] - m["ftir_ec"]
    m["hips_to_ftir_ratio"] = m["hips_bc_mac10_ugm3"] / m["ftir_ec"]
    m["oc_ec_ratio"] = m["ftir_oc"] / m["ftir_ec"]
    return m

def add_project_exclusion_flags(df):
    # Add project exclusion/outlier flags using research/ftir_hips_chem/scripts/outliers.py.
    parts = []
    for site in SITE_CODES:
        site_df = df[df["site"] == site].copy()
        if site_df.empty:
            continue
        flag_input = site_df.copy()
        flag_input["filter_id"] = flag_input["base_filter_id"]
        flag_input["aeth_bc"] = flag_input["aeth_ir_ugm3"] * 1000.0
        flag_input["filter_ec"] = flag_input["ftir_ec"] * 1000.0
        flag_input = apply_exclusion_flags(flag_input, site)
        flag_input = apply_threshold_flags(flag_input, site)
        site_df["is_excluded"] = flag_input["is_excluded"].values
        site_df["exclusion_reason"] = flag_input.get("exclusion_reason", "").values
        site_df["is_outlier"] = flag_input["is_outlier"].values
        site_df["outlier_reason"] = flag_input.get("outlier_reason", "").values
        parts.append(site_df)
    out = pd.concat(parts, ignore_index=True) if parts else df.copy()
    out["is_clean"] = ~(out.get("is_excluded", False) | out.get("is_outlier", False))
    return out

def regression_row(df, x, y, label):
    d = df[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < 3:
        return {"label": label, "x": x, "y": y, "n": len(d), "slope": np.nan, "intercept": np.nan, "r2": np.nan, "p": np.nan}
    lr = stats.linregress(d[x], d[y])
    return {"label": label, "x": x, "y": y, "n": len(d), "slope": lr.slope, "intercept": lr.intercept, "r2": lr.rvalue**2, "p": lr.pvalue}

def save_table(df, name):
    path = OUT_DIR / name
    df.to_csv(path, index=False)
    print(f"Wrote {path}")
    return path


In [3]:
m = matched_dataset(include_chem=False)
required = ["ftir_ec", "hips_fabs", "aeth_ir_ugm3"]
m["has_three_methods"] = m[required].notna().all(axis=1)

coverage = (
    m.groupby("site")
     .agg(
         filters=("base_filter_id", "nunique"),
         ec_hips=("hips_fabs", lambda s: int((s.notna() & m.loc[s.index, "ftir_ec"].notna()).sum())),
         three_method=("has_three_methods", "sum"),
         aeth_missing=("aeth_ir_ugm3", lambda s: int(s.isna().sum())),
     )
     .reset_index()
)
save_table(coverage, "matched_three_method_coverage.csv")
coverage


Loaded filter data: /Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem/Filter Data/unified_filter_dataset.pkl  rows=44,493
Wrote /Users/ahmadjalil/github/aethmodular/research/catch_up/output/01_matched_three_method_exclusion_sensitivity/matched_three_method_coverage.csv


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,site,filters,ec_hips,three_method,aeth_missing
0,Addis_Ababa,224,190,173,51
1,Beijing,222,163,65,157
2,Delhi,120,63,24,83
3,JPL,184,130,65,119


In [4]:
rows = []
comparisons = [
    ("ftir_ec", "hips_fabs", "HIPS_Fabs_vs_FTIR_EC"),
    ("ftir_ec", "aeth_ir_ugm3", "Aeth_IR_vs_FTIR_EC"),
    ("aeth_ir_ugm3", "hips_fabs", "HIPS_Fabs_vs_Aeth_IR"),
]

for site in SITE_CODES:
    site_df = m[m["site"] == site].copy()
    flag_input = site_df.copy()
    flag_input["filter_id"] = flag_input["base_filter_id"]
    flag_input["aeth_bc"] = flag_input["aeth_ir_ugm3"] * 1000.0
    flag_input["filter_ec"] = flag_input["ftir_ec"] * 1000.0
    flag_input = apply_exclusion_flags(flag_input, site)
    flag_input = apply_threshold_flags(flag_input, site)
    site_df["is_excluded"] = flag_input["is_excluded"].values
    site_df["is_outlier"] = flag_input["is_outlier"].values
    site_df["exclusion_reason"] = flag_input.get("exclusion_reason", "").values
    site_df["outlier_reason"] = flag_input.get("outlier_reason", "").values
    for x, y, label in comparisons:
        rows.append(regression_row(site_df, x, y, f"{site} | raw | {label}"))
        rows.append(regression_row(site_df[site_df["has_three_methods"]], x, y, f"{site} | matched_three_method | {label}"))
        rows.append(regression_row(site_df[~site_df["is_excluded"] & ~site_df["is_outlier"]], x, y, f"{site} | exclusions_applied | {label}"))
        rows.append(regression_row(site_df[site_df["has_three_methods"] & ~site_df["is_excluded"] & ~site_df["is_outlier"]], x, y, f"{site} | matched_three_method_exclusions | {label}"))

stats_table = pd.DataFrame(rows)
save_table(stats_table, "crossplot_stats_exclusion_sensitivity.csv")
stats_table


Wrote /Users/ahmadjalil/github/aethmodular/research/catch_up/output/01_matched_three_method_exclusion_sensitivity/crossplot_stats_exclusion_sensitivity.csv


,label,x,y,n,slope,intercept,r2,p
0,Beijing | raw | HIPS_Fabs_vs_FTIR_EC,ftir_ec,hips_fabs,163,5.590091,6.212398,0.543052,3.527466e-29
1,Beijing | matched_three_method | HIPS_Fabs_vs_...,ftir_ec,hips_fabs,65,6.063104,5.126623,0.719015,5.048179e-19
2,Beijing | exclusions_applied | HIPS_Fabs_vs_FT...,ftir_ec,hips_fabs,163,5.590091,6.212398,0.543052,3.527466e-29
3,Beijing | matched_three_method_exclusions | HI...,ftir_ec,hips_fabs,65,6.063104,5.126623,0.719015,5.048179e-19
4,Beijing | raw | Aeth_IR_vs_FTIR_EC,ftir_ec,aeth_ir_ugm3,65,0.587569,0.320608,0.566039,4.998528e-13
5,Beijing | matched_three_method | Aeth_IR_vs_FT...,ftir_ec,aeth_ir_ugm3,65,0.587569,0.320608,0.566039,4.998528e-13
6,Beijing | exclusions_applied | Aeth_IR_vs_FTIR_EC,ftir_ec,aeth_ir_ugm3,65,0.587569,0.320608,0.566039,4.998528e-13
7,Beijing | matched_three_method_exclusions | Ae...,ftir_ec,aeth_ir_ugm3,65,0.587569,0.320608,0.566039,4.998528e-13
8,Beijing | raw | HIPS_Fabs_vs_Aeth_IR,aeth_ir_ugm3,hips_fabs,65,8.129316,4.498780,0.788365,6.403218e-23
9,Beijing | matched_three_method | HIPS_Fabs_vs_...,aeth_ir_ugm3,hips_fabs,65,8.129316,4.498780,0.788365,6.403218e-23


In [5]:
excluded_rows = []
for site in SITE_CODES:
    site_df = m[m["site"] == site].copy()
    flag_input = site_df.copy()
    flag_input["filter_id"] = flag_input["base_filter_id"]
    flag_input["aeth_bc"] = flag_input["aeth_ir_ugm3"] * 1000.0
    flag_input["filter_ec"] = flag_input["ftir_ec"] * 1000.0
    flag_input = apply_threshold_flags(apply_exclusion_flags(flag_input, site), site)
    flagged = flag_input[flag_input["is_excluded"] | flag_input["is_outlier"]].copy()
    if len(flagged):
        excluded_rows.append(flagged[["site", "date", "filter_id", "ftir_ec", "hips_fabs", "aeth_ir_ugm3", "is_excluded", "exclusion_reason", "is_outlier", "outlier_reason"]])

exclusion_audit = pd.concat(excluded_rows, ignore_index=True) if excluded_rows else pd.DataFrame()
save_table(exclusion_audit, "excluded_sample_audit.csv")
exclusion_audit


Wrote /Users/ahmadjalil/github/aethmodular/research/catch_up/output/01_matched_three_method_exclusion_sensitivity/excluded_sample_audit.csv


,site,date,filter_id,ftir_ec,hips_fabs,aeth_ir_ugm3,is_excluded,exclusion_reason,is_outlier,outlier_reason
0,Delhi,2024-06-21,INDH-0172,60.640447,225.518313,NaN,False,,True,high_ec(>20000);
1,JPL,2022-10-08,USPA-0158,1.016118,8.601224,NaN,False,,True,high_either;
2,JPL,2023-07-05,USPA-0261,1.143906,6.245657,0.455906,False,,True,high_either;
